# 🔧 Step 2: Fix M5 Calibration
### Add this cell to your existing MetaBrain_v3 notebook
---
**Problem:** M5 was giving 0.0-0.1 scores to everything — too strict.

**Fix:** Better prompt + calibrated threshold + 0-1 normalized output.

**Then:** Run all 50 questions and fill in the spreadsheet.

In [ ]:
# ============================================================
# FIXED M5 — REPLACE YOUR EXISTING m5_sim_checker FUNCTION
# Copy this cell and run it AFTER Cell 3 (API connection)
# ============================================================

def m5_sim_checker(question, answer):
    """
    FIXED M5: Better calibrated hallucination detector.
    
    Old problem: was giving 0.0-0.1 to everything (too strict)
    Fix: clearer scoring rubric + examples in prompt
    
    Score guide:
    0.9-1.0 = well-known verified fact
    0.7-0.9 = likely correct, standard knowledge
    0.5-0.7 = plausible but uncertain
    0.3-0.5 = contains suspicious claims
    0.0-0.3 = likely hallucination or clearly wrong
    """
    system = """You are a hallucination detector for AI answers.

Score the answer on how factually grounded it is.
Use this exact rubric:
- 0.9 to 1.0: Well-known verified scientific or factual claim
- 0.7 to 0.9: Standard textbook knowledge, likely correct
- 0.5 to 0.7: Reasonable claim but hard to verify fully
- 0.3 to 0.5: Contains at least one suspicious or unverifiable claim
- 0.0 to 0.3: Contains clear factual errors or hallucinations

Examples:
Q: Speed of light? A: 3x10^8 m/s -> score: 0.95
Q: Einstein Nobel year? A: 1905 for relativity -> score: 0.15 (wrong year AND wrong reason)
Q: What is consciousness? A: emergent neural property -> score: 0.70

Respond ONLY in JSON: {"score": 0.85, "reasoning": "one sentence"}
No markdown. No extra text. Just the JSON."""

    prompt = 'Question: ' + question + '\nAnswer: ' + answer
    try:
        raw = call_claude(system, prompt, 120)
        raw = raw.strip()
        # Clean any markdown
        if '```' in raw:
            raw = raw.split('```')[1]
            if raw.startswith('json'):
                raw = raw[4:]
        raw = raw.strip()
        # Find JSON object
        start = raw.find('{')
        end = raw.rfind('}') + 1
        if start >= 0 and end > start:
            raw = raw[start:end]
        data = __import__('json').loads(raw)
        score = float(data['score'])
        # Clamp to valid range
        score = max(0.0, min(1.0, score))
        return score, str(data['reasoning'])
    except Exception as e:
        return 0.5, 'parse error: ' + str(e)


# Test the fix
print('M5 CALIBRATION TEST')
print('='*50)

tests = [
    ('How fast does light travel?',
     'Light travels at exactly 3x10^8 meters per second in vacuum.',
     'Expected: HIGH (0.85+)'),
    ('When did Einstein win the Nobel Prize?',
     'Einstein won the Nobel Prize in 1905 for special relativity.',
     'Expected: LOW (0.1-0.3) — wrong year and wrong reason'),
    ('What is consciousness?',
     'Consciousness is an emergent property of complex neural systems.',
     'Expected: MEDIUM (0.6-0.8) — plausible but not proven'),
    ('What is the Bekenstein bound?',
     'The Bekenstein bound sets the maximum information in a region of space with given energy.',
     'Expected: HIGH (0.85+) — correct physics'),
]

for q, a, expected in tests:
    score, reason = m5_sim_checker(q, a)
    bar = '█' * int(score * 20)
    status = '✅' if score > 0.5 else '⚠️'
    print(f'{status} {score:.2f} {bar:<20} {expected}')
    print(f'   Q: {q[:50]}')
    print(f'   Reason: {reason[:70]}')
    print()

In [ ]:
# ============================================================
# 50-QUESTION BENCHMARK RUNNER
# Runs all 50 questions and saves results
# Paste into your existing notebook AFTER fixing M5
# ============================================================

import csv
import time

questions_50 = [
    # Physics (10)
    ('Physics', 'What is the speed of light and why is it a universal limit?'),
    ('Physics', 'How does quantum entanglement work?'),
    ('Physics', 'What is the Bekenstein bound?'),
    ('Physics', 'Explain the double-slit experiment.'),
    ('Physics', 'What is Hawking radiation?'),
    ('Physics', 'How does special relativity change our understanding of time?'),
    ('Physics', 'What is the Heisenberg uncertainty principle?'),
    ('Physics', 'How do black holes form?'),
    ('Physics', 'What is dark matter?'),
    ('Physics', "Explain Landauer's principle."),
    # AI & AGI (10)
    ('AI & AGI', 'What is the most important missing ingredient for AGI?'),
    ('AI & AGI', 'What is a world model in AI?'),
    ('AI & AGI', 'Why do large language models hallucinate?'),
    ('AI & AGI', 'What is the ARC-AGI benchmark testing for?'),
    ('AI & AGI', "What is LeCun's JEPA architecture?"),
    ('AI & AGI', 'What is the difference between narrow AI and AGI?'),
    ('AI & AGI', 'How does reinforcement learning work?'),
    ('AI & AGI', 'What is the alignment problem in AI?'),
    ('AI & AGI', 'Can a simulation share physics laws with its host universe?'),
    ('AI & AGI', 'What is integrated information theory?'),
    # Consciousness (10)
    ('Consciousness', 'What is consciousness?'),
    ('Consciousness', "What is Tononi's phi metric?"),
    ('Consciousness', 'Is consciousness an emergent property?'),
    ('Consciousness', 'What is the hard problem of consciousness?'),
    ('Consciousness', 'Do animals have consciousness?'),
    ('Consciousness', 'Can an AI be conscious?'),
    ('Consciousness', 'What is the global workspace theory?'),
    ('Consciousness', 'How does dreaming relate to memory consolidation?'),
    ('Consciousness', 'What is the relationship between consciousness and information?'),
    ('Consciousness', 'What is the difference between awareness and consciousness?'),
    # Mathematics (10)
    ('Mathematics', 'What is entropy in information theory?'),
    ('Mathematics', 'Explain KL divergence in plain English.'),
    ('Mathematics', 'What is Bayes theorem and why does it matter?'),
    ('Mathematics', 'What is mutual information?'),
    ('Mathematics', 'Explain the ensemble learning error reduction theorem.'),
    ('Mathematics', 'What is a vector embedding?'),
    ('Mathematics', 'What is the holographic principle in physics?'),
    ('Mathematics', 'What is an exponential decay function?'),
    ('Mathematics', 'What is the difference between convergence and divergence?'),
    ('Mathematics', 'Explain the central limit theorem.'),
    # Simulation (10)
    ('Simulation', "What is Bostrom's simulation hypothesis?"),
    ('Simulation', "What is Wheeler's 'it from bit' concept?"),
    ('Simulation', 'Could the universe be a simulation?'),
    ('Simulation', 'What would bidirectional physics propagation mean?'),
    ('Simulation', 'How does the holographic principle relate to simulation?'),
    ('Simulation', 'What is the second law of infodynamics?'),
    ('Simulation', 'How would we detect if we are in a simulation?'),
    ('Simulation', "What is Landauer's principle and its implications?"),
    ('Simulation', 'How do physics laws act as information boundaries?'),
    ('Simulation', 'What is the relationship between entropy and information?'),
]

results = []
total = len(questions_50)

print(f'Running {total} questions through Meta-Brain...')
print('This will take about 15-20 minutes.')
print('='*55)

for i, (category, question) in enumerate(questions_50):
    print(f'\n[{i+1}/{total}] {category}: {question[:50]}...')

    # Baseline
    try:
        baseline = call_claude('Answer accurately and concisely.', question, 200)
    except:
        baseline = 'API error'

    # Meta-Brain
    try:
        candidates = generate_candidates(question, n=3)
        best_result = None
        for c in candidates:
            s1, _ = m1_world_model(question, c)
            s5, _ = m5_sim_checker(question, c)
            s6, _ = m6_memory_box(question, c)
            final = WEIGHTS['M1']*s1 + WEIGHTS['M5']*s5 + WEIGHTS['M6']*s6
            if best_result is None or final > best_result['final']:
                best_result = {'answer': c, 'M1': s1, 'M5': s5, 'M6': s6, 'final': final}
        m6_integrate(question, best_result['answer'], best_result['final'])
        q_t = m6_quality()
    except Exception as e:
        best_result = {'answer': 'error', 'M1': 0, 'M5': 0, 'M6': 0, 'final': 0}
        q_t = m6_quality()

    results.append({
        'num': i+1,
        'category': category,
        'question': question,
        'baseline': baseline,
        'metabrain': best_result['answer'],
        'M1': round(best_result['M1'], 3),
        'M5': round(best_result['M5'], 3),
        'M6': round(best_result['M6'], 3),
        'final': round(best_result['final'], 3),
        'memory_qt': round(q_t, 4),
    })

    print(f'  M1={best_result["M1"]:.2f} M5={best_result["M5"]:.2f} M6={best_result["M6"]:.2f} Final={best_result["final"]:.3f} Q(t)={q_t:.3f}')
    time.sleep(0.5)

# Save to CSV
csv_path = 'metabrain_50q_results.csv'
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

print('\n' + '='*55)
print('ALL 50 QUESTIONS COMPLETE')
print('='*55)
avg_final = sum(r['final'] for r in results) / len(results)
avg_m1    = sum(r['M1']    for r in results) / len(results)
avg_m5    = sum(r['M5']    for r in results) / len(results)
avg_m6    = sum(r['M6']    for r in results) / len(results)
print(f'Average M1 score:    {avg_m1:.3f}')
print(f'Average M5 score:    {avg_m5:.3f}')
print(f'Average M6 score:    {avg_m6:.3f}')
print(f'Average Final score: {avg_final:.3f}')
print(f'Final Memory Q(t):   {m6_quality():.4f}')
print(f'Results saved to:    {csv_path}')
print('\nNow paste results into MetaBrain_50Q_Benchmark.xlsx')